In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("heart_failure_clinical_records_dataset.csv")
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


In [5]:
df.duplicated().sum()

np.int64(0)

In [10]:
for a in df.columns:
    print("__________________" , a)
    print(df[a].unique())
    print("________End________")

__________________ age
[75.    55.    65.    50.    90.    60.    80.    62.    45.    49.
 82.    87.    70.    48.    68.    53.    95.    58.    94.    85.
 69.    72.    51.    57.    42.    41.    67.    79.    59.    44.
 63.    86.    66.    43.    46.    61.    81.    52.    64.    40.
 60.667 73.    77.    78.    54.    47.    56.   ]
________End________
__________________ anaemia
[0 1]
________End________
__________________ creatinine_phosphokinase
[ 582 7861  146  111  160   47  246  315  157  123   81  231  981  168
   80  379  149  125   52  128  220   63  148  112  122   60   70   23
  249  159   94  855 2656  235  124  571  127  588 1380  553  129  577
   91 3964   69  260  371   75  607  789  364 7702  318  109   68  250
  110  161  113 5882  224   92  102  203  336   76   55  280   78   84
  115   66  897  154  144  133  514   59  156   61  305  898 5209   53
  328  748 1876  936  292  369  143  754  400   96  737  358  200  248
  270 1808 1082  719  193 4540  646  281

In [11]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [12]:
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

In [13]:
y.value_counts()

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

In [20]:
y_train_sm.value_counts()

DEATH_EVENT
0    162
1    162
Name: count, dtype: int64

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [22]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = []

In [23]:
for name, model in models.items():
    model.fit(X_train_scaled, y_train_sm)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.82      0.88      0.85        41
           1       0.69      0.58      0.63        19

    accuracy                           0.78        60
   macro avg       0.75      0.73      0.74        60
weighted avg       0.78      0.78      0.78        60

--------------------------------------------------
=== Decision Tree ===
              precision    recall  f1-score   support

           0       0.88      0.90      0.89        41
           1       0.78      0.74      0.76        19

    accuracy                           0.85        60
   macro avg       0.83      0.82      0.82        60
weighted avg       0.85      0.85      0.85        60

--------------------------------------------------
=== Random Forest ===
              precision    recall  f1-score   support

           0       0.84      0.93      0.88        41
           1       0.80      0.63      0.71        19

    accu

In [26]:
results_df = pd.DataFrame(results)
print("\n=== Model Comparison Table ===")
print(results_df)


=== Model Comparison Table ===
                 Model  Accuracy  Precision  Recall  F1-Score
0  Logistic Regression    0.7833     0.6875  0.5789    0.6286
1        Decision Tree    0.8500     0.7778  0.7368    0.7568
2        Random Forest    0.8333     0.8000  0.6316    0.7059


In [29]:
confusion_matrix(y_test, y_pred)

array([[38,  3],
       [ 7, 12]])

In [30]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, f1_score

In [31]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

In [32]:
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)

In [33]:
grid_search.fit(X_train_scaled, y_train_sm)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [34]:
best_rf = grid_search.best_estimator_

In [35]:
y_pred_tuned = best_rf.predict(X_test_scaled)

In [36]:
print("=== Best Hyperparameters ===")
print(grid_search.best_params_)

print("\n=== Final Tuned Model Metrics ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned) * 100:.2f}%")
print(f"Recall:   {recall_score(y_test, y_pred_tuned) * 100:.2f}%")
print(f"F1-Score: {f1_score(y_test, y_pred_tuned) * 100:.2f}%")

=== Best Hyperparameters ===
{'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

=== Final Tuned Model Metrics ===
Accuracy: 83.33%
Recall:   63.16%
F1-Score: 70.59%


In [37]:
print("\n=== Tuned Classification Report ===")
print(classification_report(y_test, y_pred_tuned))

print("\n=== Final Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_tuned))


=== Tuned Classification Report ===
              precision    recall  f1-score   support

           0       0.84      0.93      0.88        41
           1       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60


=== Final Confusion Matrix ===
[[38  3]
 [ 7 12]]
